In [1]:
import torch
from torch import nn
from d2l import torch as d2l

In [2]:
def nin_block(out_channels, kernel_size, strides, padding):
    return nn.Sequential(
        nn.LazyConv2d(out_channels, kernel_size, strides, padding),
        nn.LazyConv2d(out_channels, kernel_size=1), nn.ReLU(),
        nn.LazyConv2d(out_channels, kernel_size=1), nn.ReLU(),
    )

In [14]:
class NiN(d2l.Classifier):
    def __init__(self, lr =0.01, num_classes=10):
        super().__init__()
        self.save_hyperparameters()
        self.net = nn.Sequential(
            # nin_block(96, kernel_size=11, strides=4, padding=0),
            # nn.MaxPool2d(kernel_size=3, stride=2),
            nin_block(256, kernel_size=5, strides=1, padding=2),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nin_block(384, kernel_size=3, strides=1, padding=1),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Dropout(0.5),
            nin_block(num_classes, kernel_size=3, strides=1, padding=1),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
        )
        self.net.apply(d2l.init_cnn)

In [ ]:
NiN().layer_summary((1,1,224,224))

In [15]:
model = NiN(lr=0.01)
trainer = d2l.Trainer(max_epochs = 10)
data = d2l.FashionMNIST(batch_size=128)
model.apply_init([next(iter(data.get_dataloader(True)))[0]], d2l.init_cnn)

In [16]:
trainer.fit(model, data)

KeyboardInterrupt: 